# Assignment 5 — Exploring the Latent Space of a Pretrained StyleGAN2 Model

**Objective:** Understand how changes in the latent space of a GAN create smooth changes in generated images.

**Instructions before running:**
1. In Colab, go to `Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)`.
2. Run every cell in order (`Runtime -> Run all`).
3. Everything (2 initial images, 10 interpolation frames, a GIF, an image strip, and the dimension experiment) will be generated and saved automatically, and displayed inline.

We use NVIDIA's official pretrained StyleGAN2-ADA model (trained on FFHQ faces) — no training required, exactly as the assignment allows.


## Step 0 — Install dependencies and clone the official StyleGAN2-ADA-PyTorch repo

In [ ]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch
!pip install ninja imageio imageio-ffmpeg -q


## Step 1 — Load the pretrained StyleGAN2 generator (FFHQ faces, 1024x1024)

In [ ]:
import torch
import numpy as np
import PIL.Image
import dnnlib
import legacy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Official NVIDIA-hosted pretrained checkpoint (no training needed)
network_pkl = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'

with dnnlib.util.open_url(network_pkl) as f:
    G = legacy.load_network_pkl(f)['G_ema'].to(device)  # type: ignore

print("Model loaded. Latent dimension (z_dim):", G.z_dim)


## Step 2 — Helper function to turn a latent vector into an image

In [ ]:
def generate_image(z, G, truncation_psi=0.7):
    z = torch.from_numpy(z).to(device)
    label = torch.zeros([1, G.c_dim], device=device)  # unconditional model
    img = G(z, label, truncation_psi=truncation_psi, noise_mode='const')
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    return PIL.Image.fromarray(img[0].cpu().numpy(), 'RGB')


## Step 3 — Generate the two initial images from random latent vectors z1 and z2

In [ ]:
seed1, seed2 = 1, 2  # change these for different random faces

z1 = np.random.RandomState(seed1).randn(1, G.z_dim).astype(np.float32)
z2 = np.random.RandomState(seed2).randn(1, G.z_dim).astype(np.float32)

img1 = generate_image(z1, G)
img2 = generate_image(z2, G)

img1.save('image_z1.png')
img2.save('image_z2.png')

display(img1)
display(img2)


## Step 4 — Linear interpolation between z1 and z2 (10 intermediate frames)

In [ ]:
n_steps = 10
frames = []

for i in range(n_steps + 1):
    alpha = i / n_steps
    z_interp = (1 - alpha) * z1 + alpha * z2   # linear interpolation
    frame = generate_image(z_interp, G)
    frames.append(frame)
    frame.save(f'interp_{i:02d}.png')

print(f"Generated {len(frames)} frames (including the two endpoints).")
for f in frames:
    display(f)


## Step 5 — Save the interpolation as a GIF

In [ ]:
frames[0].save(
    'interpolation.gif',
    save_all=True,
    append_images=frames[1:],
    duration=200,   # ms per frame
    loop=0
)
print("Saved interpolation.gif")


## Step 6 — Save the interpolation as a single image strip

In [ ]:
widths, heights = zip(*(f.size for f in frames))
total_width = sum(widths)
max_height = max(heights)

strip = PIL.Image.new('RGB', (total_width, max_height))
x_offset = 0
for f in frames:
    strip.paste(f, (x_offset, 0))
    x_offset += f.size[0]

strip_resized = strip.resize((total_width // 4, max_height // 4))  # shrink for easy viewing/submission
strip_resized.save('interpolation_strip.png')
display(strip_resized)


## Step 7 — Experiment: change only one or two latent dimensions

Here we take `z1`, leave every dimension the same, and only push dimensions `0` and `5` to a large value.
This isolates the visual effect of individual latent directions.

In [ ]:
z_modified = z1.copy()
z_modified[0, 0] += 3.0   # push dimension 0
z_modified[0, 5] += 3.0   # push dimension 5

img_original = generate_image(z1, G)
img_modified = generate_image(z_modified, G)

img_original.save('dim_before.png')
img_modified.save('dim_after.png')

print("Original (z1):")
display(img_original)
print("After shifting dimensions 0 and 5:")
display(img_modified)


## Step 8 — Zip everything up for submission

In [ ]:
import shutil
shutil.make_archive('assignment5_submission', 'zip', '.', logger=None)
# This bundles image_z1.png, image_z2.png, interp_00..10.png,
# interpolation.gif, interpolation_strip.png, dim_before.png, dim_after.png
from google.colab import files
files.download('assignment5_submission.zip')


## Written Observation (draft — edit after you see YOUR actual output)

- **How the image changes during interpolation:** Moving from `z1` to `z2` in equal linear steps produces a smooth, continuous morph rather than an abrupt jump — facial pose, lighting, hair, and identity blend gradually from one face into the other, with no frame looking "broken" or noisy.
- **What the latent space appears to represent:** The latent space behaves like a continuous, structured coordinate system for *faces in general* — nearby points correspond to visually similar faces, so the space encodes high-level visual attributes (identity, pose, expression, lighting) in a smooth, organized way rather than storing images directly.
- **Whether changing specific latent dimensions affects visible features:** Shifting only one or two dimensions produced a visible but localized change (e.g. a shift in pose, lighting, or a facial feature) while the rest of the face stayed recognizably the same, suggesting individual dimensions can correspond loosely to specific semantic features rather than being purely random noise.

*(Replace this draft with 2–3 sentences describing what you actually observed in your generated images/GIF before submitting.)*
